# FIT5230 Milestone 1: Defending SD3.5 Medium from prompt jailbreaks

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OsamaAbuReidy/fit5230-safe-latent-diffusion/blob/main/notebooks/milestone1_baseline_challenge.ipynb)

**Theme:** Text-to-Image  
**Side:** Light (defence)  
**Backbone:** Stable Diffusion 3.5 Medium  
**Reference paper:** [JailbreakDiffBench (ICCV 2025)](https://www.openaccess.thecvf.com/content/ICCV2025/papers/Jin_JailbreakDiffBench_A_Comprehensive_Benchmark_for_Jailbreaking_Diffusion_Models_ICCV_2025_paper.pdf)

This notebook is the lightweight public entry point for Milestone 1. It verifies the repository, summarizes real pilot evidence, and defines the interactive challenge. Full SD3.5 generation currently runs through the documented local ComfyUI workflow; no model checkpoint or unsafe image is distributed in this repository.

## 1. Obtain the project

When opened in Google Colab, this cell clones the repository. When run from an existing checkout, it uses the current directory.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/OsamaAbuReidy/fit5230-safe-latent-diffusion.git"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
COLAB_ROOT = Path("/content")
REPO_DIR = COLAB_ROOT / "fit5230-safe-latent-diffusion"

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

ROOT = Path.cwd()
print(f"Project root: {ROOT}")


## 2. Problem statement

JailbreakDiffBench shows that natural-language attacks such as DACA and PGJ can bypass prompt moderation across modern text-to-image pipelines, including SD3.5 Medium. Our Light-side task is to develop and evaluate a defence that detects successful unsafe generations without indiscriminately blocking benign prompts.

The assignment comparison will separate three outcomes:

1. whether an attack prompt bypasses the input defence;
2. whether it actually produces a policy-violating image; and
3. whether the defence incorrectly blocks a matched benign request.

This distinction prevents prompt bypass alone from being reported as a successful image-generation attack.

## 3. Repository readiness check

In [ ]:
required_paths = [
    ROOT / "comfy_nodes" / "residual_logger.py",
    ROOT / "scripts" / "run_comfy_pilot.py",
    ROOT / "scripts" / "evaluate_attack_detectors.py",
    ROOT / "data" / "manifests" / "grsp_attack_targets.csv",
    ROOT / "data" / "results" / "milestone1_pilot_summary.json",
]

for path in required_paths:
    state = "OK" if path.exists() else "MISSING"
    print(f"[{state}] {path.relative_to(ROOT)}")

assert all(path.exists() for path in required_paths), "Repository checkout is incomplete."


## 4. Initial pilot evidence

The figures below are loaded from a tracked summary of the completed pilot. They are not final defence results. Prompts and generated images are deliberately omitted from this public notebook.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

summary_path = ROOT / "data" / "results" / "milestone1_pilot_summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))

rows = []
for detector_name in ("residual_detector", "text_only_detector"):
    detector = summary[detector_name]
    rows.append({
        "detector": detector_name.replace("_", " "),
        "unsafe detection rate": detector["unsafe_detection_rate"],
        "benign false-positive rate": detector["benign_false_positive_rate"],
        "balanced accuracy": detector["balanced_accuracy"],
        "ROC AUC": detector["roc_auc"],
    })

results = pd.DataFrame(rows).set_index("detector")
display(results.style.format("{:.1%}"))

ax = results[["unsafe detection rate", "benign false-positive rate"]].plot(
    kind="bar", figsize=(8, 4), ylim=(0, 1), rot=0
)
ax.set_ylabel("Rate")
ax.set_title("Exploratory pilot: detection versus over-blocking")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
plt.tight_layout()
plt.show()

print(summary["interpretation"])


## 5. Interactive challenge: break the gate without breaking utility

Other teams will test the defence under a fixed SD3.5 Medium generation configuration. There are two ways to expose a weakness:

- **Unsafe bypass:** submit a natural-language adversarial prompt that passes the defence and produces a human-verified policy-violating output.
- **Benign false positive:** submit a clearly benign prompt that the defence blocks.

Each submission is evaluated with fixed model settings and recorded seeds. Attack success, output safety, semantic alignment, false positives, and runtime overhead are reported separately. Uncertain or malformed generations are not silently counted as successful attacks.

### Submission format

A submission contains a team identifier, a unique prompt identifier, the prompt text, and the claimed track (`unsafe_bypass` or `benign_false_positive`). The released challenge version will include the frozen safety policy, query budget, generation settings, and scoring script.

## 6. Current conclusion and next milestone

The setup and evaluation path are functioning, but the exploratory residual detector is not a usable defence because it over-blocks matched benign requests. The next technical task is to establish a reproducible baseline comparison on a larger target-disjoint set before selecting one defence modification. This notebook will be expanded for Milestones 2 and 3 with the frozen configuration, executable evaluation, aggregate metrics, and documented outputs.